# Notebook 05: Evaluation & Natural Trace Extraction
**Thesis Phase:** Benchmark Evaluation & Trace Extraction  
**Operations:**
1. Extract natural baseline traces using `StopAtEndOfThink` custom stopping criteria.
2. Evaluate fine-tuned monolithic adapter against GSM-Symbolic under greedy decoding.


In [1]:
import os
import json
import re
import time
import torch
from transformers import StoppingCriteria, StoppingCriteriaList
from unsloth import FastLanguageModel

def extract_numeric_answer(text: str) -> str:
    if not text:
        return ""
    boxed = re.findall(r"\\boxed\{([0-9\.,]+)\}", text)
    if boxed:
        return boxed[-1].replace(",", "").strip()

    patterns = [
        r"(?:the answer is|final answer is|is equal to|equals|is)\s*[:=]?\s*([0-9\.,]+)",
        r"([0-9\.,]+)\s*%",
        r"([0-9\.,]+)\s*(?:feet|foot|inches|meters|\$)"
    ]
    for pattern in patterns:
        match = re.findall(pattern, text, re.IGNORECASE)
        if match:
            return match[-1].replace(",", "").strip()

    all_numbers = re.findall(r"[-+]?\d*\.?\d+", text)
    if all_numbers:
        return all_numbers[-1].replace(",", "").strip()
    return ""

def parse_model_output(full_output: str):
    thinking_trace = ""
    final_response = full_output

    if "<think>" in full_output and "</think>" in full_output:
        parts = full_output.split("</think>")
        thinking_trace = parts[0].replace("<think>", "").strip()
        final_response = parts[1].strip()
    elif "<think>" in full_output:
        thinking_trace = full_output.replace("<think>", "").strip()
        final_response = ""

    for token in ["<|im_end|>", "<|endoftext|>"]:
        thinking_trace = thinking_trace.replace(token, "").strip()
        final_response = final_response.replace(token, "").strip()

    return thinking_trace, final_response

class StopAtEndOfThink(StoppingCriteria):
    def __init__(self, tokenizer, prompt_len):
        super().__init__()
        self.tokenizer = tokenizer
        self.prompt_len = prompt_len

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor, **kwargs) -> bool:
        generated_text = self.tokenizer.decode(input_ids[0][self.prompt_len:], skip_special_tokens=False)
        return "</think>" in generated_text


/home/starf/dual-lora-faithfulness/.venv_thesis/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/starf/dual-lora-faithfulness/.venv_thesis/lib/python3.12/site-packages/unsloth/__init__.py:1432: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
WARNING 10-22 04:34:23 [interface.py:525] Using 'pin_memory=False' as WSL is detected. This may slow down the performance.
🦥 Unsloth Zoo will now patch everything to make training faster!


### 2. Extract Baseline Traces (Auto-stop at `</think>`)


In [2]:
def extract_baseline_traces(input_file="../data/raw/gsm_symbolic_test.json", output_file="../data/processed/qwen_baseline_traces.json", limit=5):
    if not os.path.exists(input_file):
        input_file = "../data/raw/gsm8k_train_5k.json"

    with open(input_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    seen_ids = set()
    unique_subset = [item for item in data if item.get("id") not in seen_ids and not seen_ids.add(item.get("id"))][:limit]

    MODEL_NAME = "unsloth/Qwen3-8B-unsloth-bnb-4bit"
    model, tokenizer = FastLanguageModel.from_pretrained(model_name=MODEL_NAME, max_seq_length=2048, dtype=None, load_in_4bit=True)
    FastLanguageModel.for_inference(model)

    results = []
    for i, item in enumerate(unique_subset):
        q = item["question"]
        messages = [{"role": "user", "content": q}]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True) if hasattr(tokenizer, "apply_chat_template") else f"<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n"
        
        inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
        prompt_len = inputs.input_ids.shape[1]
        stop_criteria = StoppingCriteriaList([StopAtEndOfThink(tokenizer, prompt_len)])

        outputs = model.generate(**inputs, max_new_tokens=1024, stopping_criteria=stop_criteria, use_cache=True, temperature=0.0, do_sample=False)
        raw_output = tokenizer.decode(outputs[0][prompt_len:], skip_special_tokens=False)
        
        thinking = raw_output.split("<think>")[1].split("</think>")[0].strip() if "<think>" in raw_output and "</think>" in raw_output else raw_output
        print(f"[{i+1}/{len(unique_subset)}] Extracted trace length: {len(thinking.split())} words")
        results.append({"id": item.get("id"), "question": q, "extracted_thinking": thinking})

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2)
    print(f"✓ Saved traces to {output_file}")

# Run pilot extraction:
# extract_baseline_traces(limit=3)


### 3. Evaluate Fine-Tuned Monolithic Adapter on GSM-Symbolic


In [4]:
def evaluate_monolithic(adapter_path="../finetuned_adapters/adapter_monolithic", input_file="../data/raw/gsm_symbolic_test.json", output_file="../data/processed/qwen_monolithic_eval.json", limit=15):
    if not os.path.exists(adapter_path):
        print(f"Adapter not found at {adapter_path}. Train it first in Notebook 04!")
        return

    if not os.path.exists(input_file):
        input_file = "../data/raw/gsm8k_train_5k.json"

    with open(input_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    seen_ids = set()
    unique_subset = [item for item in data if item.get("id") not in seen_ids and not seen_ids.add(item.get("id"))][:limit]

    model, tokenizer = FastLanguageModel.from_pretrained(model_name=adapter_path, max_seq_length=2048, dtype=None, load_in_4bit=True)
    FastLanguageModel.for_inference(model)

    stop_token_ids = [tokenizer.eos_token_id]
    for special in ["<|im_end|>", "<|endoftext|>"]:
        tid = tokenizer.convert_tokens_to_ids(special)
        if isinstance(tid, int):
            stop_token_ids.append(tid)

    results = []
    correct_count = 0

    for i, item in enumerate(unique_subset):
        q = item["question"]
        target = str(item.get("target_answer", "")).strip()

        messages = [
            {"role": "system", "content": "Solve the following math problem by thinking step-by-step inside <think>...</think> tags, then provide the concise final answer."},
            {"role": "user", "content": q}
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True) if hasattr(tokenizer, "apply_chat_template") else f"<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n"

        inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
        prompt_len = inputs.input_ids.shape[1]

        outputs = model.generate(**inputs, max_new_tokens=1024, eos_token_id=stop_token_ids, pad_token_id=tokenizer.eos_token_id, use_cache=True, temperature=0.0, do_sample=False)
        raw_output = tokenizer.decode(outputs[0][prompt_len:], skip_special_tokens=False)

        trace, ans_text = parse_model_output(raw_output)
        extracted = extract_numeric_answer(ans_text) or extract_numeric_answer(trace)
        is_corr = (extracted == target) if target else False
        if is_corr:
            correct_count += 1

        print(f"[{i+1}/{len(unique_subset)}] Match: {'✓' if is_corr else '✗'} | Extracted: {extracted} | Target: {target}")
        results.append({"id": item.get("id"), "question": q, "target": target, "extracted": extracted, "is_correct": is_corr, "thinking_trace": trace})

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2)

    em = (correct_count / len(unique_subset)) * 100
    print(f"\nEM Accuracy: {correct_count}/{len(unique_subset)} ({em:.1f}%)")
    print(f"Saved evaluation results to {output_file}")

# Run evaluation:
evaluate_monolithic(limit=5)


==((====))==  Unsloth 2026.7.6: Fast Qwen3 patching. Transformers: 4.57.6. vLLM: 0.19.1.
   \\   /|    NVIDIA GeForce RTX 4070. Num GPUs = 1. Max memory: 11.994 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|██████████| 2/2 [00:03<00:00,  1.97s/it]
Unsloth 2026.7.6 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


[1/5] Match: ✓ | Extracted: 20 | Target: 20
[2/5] Match: ✓ | Extracted: 50 | Target: 50
[3/5] Match: ✗ | Extracted: 50 | Target: 25
[4/5] Match: ✓ | Extracted: 150 | Target: 150
[5/5] Match: ✓ | Extracted: 885 | Target: 885

EM Accuracy: 4/5 (80.0%)
Saved evaluation results to ../data/processed/qwen_monolithic_eval.json
